## Author: Gracie Grimsrud
## Date: 09/22/2026

Multi-subject version of `temp_match_overlap_singlesub.ipynb` for the discovery sample.
For each subject: consensus + per-input disagreement maps (dscalar), summary CSVs, and a surface PNG.
Then: group summary CSVs and a consensus % bar plot.

Functions live in `pfm_overlap.py` (same folder).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import nilearn
import numpy as np
import pandas as pd
from IPython.display import display

code_directory = Path(
    "/oak/stanford/groups/russpold/users/grimsrud/projects/pfm_compare/code/plot_pfm_overlap"
)

if str(code_directory) not in sys.path:
    sys.path.insert(0, str(code_directory))

import pfm_overlap as pfm

print("Python:", sys.executable)
print("nilearn:", nilearn.__version__)
print("pfm_overlap:", pfm.__file__)

In [ ]:
# inputs
base_directory = Path(
    "/oak/stanford/groups/russpold/users/grimsrud/projects/pfm_compare/analysis/temp_match_results"
)

subjects = ["sub-s03", "sub-s10", "sub-s19", "sub-s29", "sub-s43"]
session = "ses-concatenated"

common_suffix = (
    "ReproTM_template-ABCC2026-a3-9to16_refine-SCAN_minsize-30.dscalar.nii"
)

input_config = {
    "rest": {
        "results_directory": "april_2026_fmriprep_xcpd",
        "task": "rest",
    },
    "task": {
        "results_directory": "april_2026_fmriprep_xcpd",
        "task": "task",
    },
    "task_resid_raw": {
        "results_directory": "july2026_raw_resid",
        "task": "taskresidRAW",
    },
}

map_index = 0

# outputs
output_root = (
    Path("/oak/stanford/groups/russpold/users/grimsrud/projects/pfm_compare/analysis/temp_match_overlap/22Sept2026_discovery")
    / "network_overlap_outputs"
)

group_output_directory = output_root / "group"

overwrite_outputs = True

# surfaces + figure settings
left_surface_path = Path(
    "/oak/stanford/groups/russpold/users/grimsrud/data/fsLR/fs_LR.32k.L.midthickness.surf.gii"
)

right_surface_path = Path(
    "/oak/stanford/groups/russpold/users/grimsrud/data/fsLR/fs_LR.32k.R.midthickness.surf.gii"
)

surface_views = [
    ("left", "lateral", left_surface_path),
    ("left", "medial", left_surface_path),
    ("right", "lateral", right_surface_path),
    ("right", "medial", right_surface_path),
]

panel_zoom = 1.5
legend_fontsize = 13

comparison_name = "_vs_".join(
    pfm.safe_filename_component(name)
    for name in input_config
)

print("Comparison name:", comparison_name)
print("Output root:", output_root)

In [ ]:
# check every input (dscalar + dlabel) for every subject
for surface_path in (left_surface_path, right_surface_path):
    if not surface_path.is_file():
        raise FileNotFoundError(f"Missing surface:\n{surface_path}")

input_files_by_subject = {
    subject: pfm.build_input_files(
        base_directory=base_directory,
        input_config=input_config,
        subject=subject,
        session=session,
        common_suffix=common_suffix,
    )
    for subject in subjects
}

input_check_rows = []

for subject, input_files in input_files_by_subject.items():
    for input_name, dscalar_path in input_files.items():
        dlabel_path = pfm.corresponding_dlabel_path(dscalar_path)

        input_check_rows.append({
            "subject": subject,
            "input": input_name,
            "dscalar": "FOUND" if dscalar_path.is_file() else "MISSING",
            "dlabel": "FOUND" if dlabel_path.is_file() else "MISSING",
        })

input_check = pd.DataFrame(input_check_rows)

display(
    input_check.pivot(
        index="subject",
        columns="input",
        values=["dscalar", "dlabel"],
    )
)

missing = input_check[
    (input_check["dscalar"] == "MISSING")
    | (input_check["dlabel"] == "MISSING")
]

if not missing.empty:
    raise FileNotFoundError(f"Missing inputs:\n{missing}")

print("All inputs found.")

In [ ]:
# network names/colors from one dlabel; confirm every dlabel uses the same table
reference_dlabel_path = pfm.corresponding_dlabel_path(
    input_files_by_subject[subjects[0]][next(iter(input_config))]
)

network_labels, network_colors = pfm.load_label_table(
    reference_dlabel_path,
    map_index,
)

for subject, input_files in input_files_by_subject.items():
    for input_name, dscalar_path in input_files.items():
        labels, colors = pfm.load_label_table(
            pfm.corresponding_dlabel_path(dscalar_path),
            map_index,
        )

        if labels != network_labels or colors != network_colors:
            raise ValueError(
                f"Label table for {subject} {input_name} differs from "
                f"{reference_dlabel_path.name}"
            )

print("All dlabel tables match.\n")

for network_id in sorted(network_labels):
    print(
        f"{network_id}: "
        f"{network_labels[network_id]!r}, "
        f"RGBA={network_colors[network_id]}"
    )

dlabel_network_cmap, color_min, color_max = pfm.make_network_cmap(
    network_colors,
    pfm.BACKGROUND_VALUES,
)

print("\nBackground values:", sorted(pfm.BACKGROUND_VALUES))
print("Color limits:", color_min, color_max)

In [ ]:
# overlap: consensus + disagreement for each subject; save dscalars + CSVs
results = {}

for subject in subjects:
    loaded = pfm.load_assignments(
        input_files_by_subject[subject],
        map_index,
    )

    overlap = pfm.compute_overlap(
        loaded["assignments"],
        loaded["background"],
        loaded["input_names"],
        network_labels,
    )

    output_paths = pfm.save_overlap_outputs(
        overlap,
        loaded,
        subject=subject,
        comparison_name=comparison_name,
        output_directory=output_root / subject,
        overwrite=overwrite_outputs,
    )

    results[subject] = {
        "input_names": loaded["input_names"],
        "overlap": overlap,
        "paths": output_paths,
    }

    n_consensus = overlap["consensus_mask"].sum()
    n_all_assigned = overlap["all_assigned"].sum()

    print(
        f"{subject}: consensus {n_consensus:,} / {n_all_assigned:,} "
        f"assigned-in-every-input "
        f"({100 * n_consensus / n_all_assigned:.2f}%), "
        f"disagreement {overlap['disagreement_mask'].sum():,}"
    )
    print(f"  saved to {output_root / subject}")

In [ ]:
# surface figure for each subject: disagreement rows + consensus row
for subject, result in results.items():
    maps_to_plot = {
        f"{input_name} disagreement": result["paths"]["disagreement_paths"][input_name]
        for input_name in result["input_names"]
    }

    maps_to_plot["consensus"] = result["paths"]["consensus_path"]

    hemisphere_values_by_map, plotted_network_ids = pfm.load_hemisphere_values(
        maps_to_plot,
        map_index,
    )

    fig = pfm.plot_surface_grid(
        hemisphere_values_by_map,
        plotted_network_ids,
        surface_views,
        cmap=dlabel_network_cmap,
        color_min=color_min,
        color_max=color_max,
        network_labels=network_labels,
        network_colors=network_colors,
        title=subject,
        panel_zoom=panel_zoom,
        legend_fontsize=legend_fontsize,
    )

    surface_figure_path = (
        output_root
        / subject
        / f"{subject}_{comparison_name}_disagreement_consensus_surfaces.png"
    )

    fig.savefig(
        surface_figure_path,
        dpi=150,
        bbox_inches="tight",  # crop empty figure border
        pad_inches=0.05,  # margin left after cropping
    )

    result["paths"]["surface_figure_path"] = surface_figure_path

    print("Saved:", surface_figure_path)
    display(fig)

In [ ]:
# group summary CSVs: every subject stacked, with a subject column
group_output_directory.mkdir(parents=True, exist_ok=True)

group_overall_summary = pd.concat(
    [
        result["overlap"]["overall_summary"].assign(subject=subject)
        for subject, result in results.items()
    ],
    ignore_index=True,
)

group_network_summary = pd.concat(
    [
        result["overlap"]["network_summary"].assign(subject=subject)
        for subject, result in results.items()
    ],
    ignore_index=True,
)

# subject first
group_overall_summary = group_overall_summary[
    ["subject"] + [c for c in group_overall_summary.columns if c != "subject"]
]

group_network_summary = group_network_summary[
    ["subject"] + [c for c in group_network_summary.columns if c != "subject"]
]

group_overall_summary_path = (
    group_output_directory
    / f"group_{comparison_name}_overall_summary.csv"
)

group_network_summary_path = (
    group_output_directory
    / f"group_{comparison_name}_network_summary.csv"
)

group_overall_summary.to_csv(group_overall_summary_path, index=False)
group_network_summary.to_csv(group_network_summary_path, index=False)

print("Saved:")
print(group_overall_summary_path)
print(group_network_summary_path)

display(
    group_overall_summary.pivot(
        index="subject",
        columns="category",
        values="count",
    )
)

In [ ]:
# consensus % per subject
# percent of grayordinates assigned in every input that have the same network in all inputs
consensus_by_subject = pd.DataFrame([
    {
        "subject": subject,
        "consensus_count": int(result["overlap"]["consensus_mask"].sum()),
        "all_assigned_count": int(result["overlap"]["all_assigned"].sum()),
    }
    for subject, result in results.items()
])

consensus_by_subject["consensus_percent"] = (
    100
    * consensus_by_subject["consensus_count"]
    / consensus_by_subject["all_assigned_count"]
)

display(consensus_by_subject)

consensus_fig, ax = plt.subplots(figsize=(6, 3.5))

bars = ax.bar(
    consensus_by_subject["subject"],
    consensus_by_subject["consensus_percent"],
    width=0.6,
    color="#4a6fa5",
)

ax.bar_label(bars, fmt="%.1f%%", padding=3, fontsize=10)

ax.set_ylim(0, 100)
ax.set_ylabel("Consensus (% of grayordinates\nassigned in every input)")
ax.set_title(
    f"Network consensus across {', '.join(input_config)}",
    fontsize=11,
)

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)

consensus_fig.tight_layout()

consensus_figure_path = (
    group_output_directory
    / f"group_{comparison_name}_consensus_percent.png"
)

consensus_fig.savefig(consensus_figure_path, dpi=150, bbox_inches="tight")

print("Saved:", consensus_figure_path)
plt.show()